# IPMSM PyAEDT GUI Test Notebook

This notebook is intentionally short and uses the same modules as the final batch runner.

Run cells from top to bottom for a setup-only smoke test. Set `RUN_ANALYSIS_IN_NOTEBOOK = True` in the options cell when you want to watch the transient solve in the AEDT GUI.


In [1]:
from pathlib import Path
import importlib
import os
import sys

WORKSPACE = Path(r"Y:\git\pyaedt_motor")
if WORKSPACE.exists():
    os.chdir(WORKSPACE)
else:
    WORKSPACE = Path.cwd()

if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from run_ipmsm_batch import (
    Simulation,
    build_spec,
    dataframe_first_row,
    export_ppt_reports,
    summarize_transient_outputs,
)
from module.ipmsm_geometry import create_ipmsm_design
import module.ipmsm_ppt_setup as ipmsm_ppt_setup
from pyaedt_module.core import pyDesktop

ipmsm_ppt_setup = importlib.reload(ipmsm_ppt_setup)
configure_ipmsm_from_ppt = ipmsm_ppt_setup.configure_ipmsm_from_ppt
create_ppt_reports = ipmsm_ppt_setup.create_ppt_reports

WORKSPACE


WindowsPath('Y:/git/pyaedt_motor')

In [2]:
# Notebook options
SHOW_AEDT_GUI = True
RUN_ANALYSIS_IN_NOTEBOOK = True  # Change to True only when you want to run the transient solve.
NUM_CORES = 4

CASE = {
    "case_id": "notebook_test_10cycle",
    "pole_number": 8,
    "slot_number": 12,
    "symmetry_factor": 4,
    "base_rpm": 1200,
    "i_peak_a": 137.8,
    "beta_deg": 30,
    "series_turns_per_phase": 48,
    "turns_per_coil_side": 12,
    "stack_length_mm": 49.45,
    "phase_resistance_ohm": 0.01,
    "vdc_v": 200,
    "initial_position_deg": -22.5,
    "transient_periods": 10,
    "steps_per_period": 90,
}

NON_GRAPHICAL = not SHOW_AEDT_GUI
SIMULATION_DIR = WORKSPACE / "simulation"
SIMULATION_DIR.mkdir(parents=True, exist_ok=True)

CASE


{'case_id': 'notebook_test_10cycle',
 'pole_number': 8,
 'slot_number': 12,
 'symmetry_factor': 4,
 'base_rpm': 1200,
 'i_peak_a': 137.8,
 'beta_deg': 30,
 'series_turns_per_phase': 48,
 'turns_per_coil_side': 12,
 'stack_length_mm': 49.45,
 'phase_resistance_ohm': 0.01,
 'vdc_v': 200,
 'initial_position_deg': -22.5,
 'transient_periods': 10,
 'steps_per_period': 90}

In [3]:
# Start AEDT and create a fresh project.
# close_on_exit=False keeps the GUI open so you can inspect progress/results.
desktop = pyDesktop(
    version=None,
    non_graphical=NON_GRAPHICAL,
    close_on_exit=False,
    new_desktop=True,
)

sim1 = Simulation(desktop=desktop, cores=NUM_CORES)
sim1.create_simulation_name(SIMULATION_DIR)
project1 = sim1.create_project(SIMULATION_DIR)
project_path = Path(project1.path)

{
    "simulation_name": sim1.PROJECT_NAME,
    "project_path": str(project_path),
    "gui_visible": SHOW_AEDT_GUI,
}


PyAEDT INFO: Python version 3.11.14 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 18:30:03) [MSC v.1929 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.22.0.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file C:\Users\Public\Documents\ESTsoft\CreatorTemp\pyaedt_peets_d787b11d-3d5e-43a4-a145-1136fc43c2db.log is enabled.
PyAEDT INFO: Log on AEDT is disabled.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: Launching PyAEDT with gRPC plugin.
PyAEDT INFO: New AEDT session is starting on gRPC port 57123.
PyAEDT INFO: Electronics Desktop started on gRPC port: 57123 after 29.180275917053223 seconds.
PyAEDT INFO: AEDT installation Path C:\Program Files\ANSYS Inc\v252\AnsysEM
PyAEDT INFO: Ansoft.ElectronicsDesktop.2025.2 version started with process ID 61008.


{'simulation_name': 'simulation71',
 'project_path': 'Y:\\git\\pyaedt_motor\\simulation\\simulation71',
 'gui_visible': True}

In [4]:
# Build the full 360-degree IPMSM geometry.
design1, input_data, object_groups = create_ipmsm_design(project1, sim1)

{
    "design": "IPMSM",
    "objects": {key: len(value) for key, value in object_groups.items()},
    "input_preview": dataframe_first_row(input_data),
}


PyAEDT INFO: Python version 3.11.14 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 18:30:03) [MSC v.1929 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.22.0.
PyAEDT INFO: Returning found Desktop session with PID 61008!
PyAEDT INFO: Project simulation71 set to active.
PyAEDT INFO: Added design 'IPMSM' of type Maxwell 2D.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler2D class has been initialized!
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects pa

{'design': 'IPMSM',
 'objects': {'stator': 1,
  'rotor': 1,
  'shaft': 1,
  'magnets': 8,
  'windings': 24,
  'region': 1,
  'band': 1},
 'input_preview': {'slot_num': 12.0,
  'pole_num': 8.0,
  'stator_outer_radius': 126.0,
  'stator_back_yoke_thick_ratio': 0.122,
  'stator_back_yoke_thick': 15.372,
  'stator_inner_ratio': 0.435,
  'stator_inner_radius': 54.81,
  'stator_shoe_thick': 1.1,
  'stator_teeth_length_ratio': 0.885,
  'stator_teeth_length': 49.39893,
  'stator_teeth_width': 13.485962044762735,
  'stator_gap': 1.89,
  'rotator_gap': 2.56,
  'shaft_ratio': 0.522,
  'rotor_radius': 108.068,
  'shaft_radius': 56.411496,
  'magnet_shield_thick': 1.542,
  'magnet_setback_ratio': 0.174,
  'magnet_thick_ratio': 0.243,
  'magnet_height_ratio': 0.839}}

In [5]:
# Apply materials, boundaries, windings, mesh, transient setup, and pre-solve reports.
# Solve is intentionally separated so GUI/debug errors are easy to locate.
ppt_spec = build_spec(CASE, default_symmetry_factor=CASE["symmetry_factor"])

ppt_setup_result = configure_ipmsm_from_ppt(
    design1,
    object_groups=object_groups,
    spec=ppt_spec,
    operation="sin_current",
    use_periodic_boundary=False,  # Full 360 model: keep False.
    create_missing_region=True,
    create_missing_band=True,
    create_reports=RUN_ANALYSIS_IN_NOTEBOOK,
    clear_existing=True,
    analyze=False,
    cores=NUM_CORES,
)

ppt_setup_result


PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Adding new material to the Project Library: 27PNF1500_CustomCoreLoss
PyAEDT INFO: Material has been added in Desktop.
PyAEDT INFO: Adding new material to the Project Library: NdFeB_1.25T
PyAEDT INFO: Material has been added in Desktop.
PyAEDT INFO: Boundary Vector Potential VectorPotentialZero has been created.
PyAEDT INFO: Boundary Band MotionSetup1 has been created.
PyAEDT INFO: Boundary Coil PhaseA_Positive_01_01 has been created.
PyAEDT INFO: Boundary Coil PhaseA_Positive_01_02 has been created.
PyAEDT INFO: Bou

{'variables': 'applied',
 'cleanup': {'boundaries': [],
  'mesh': [],
  'motion': [],
  'setup': [],
  'reports': []},
 'region_band': {'region': ['Region'], 'band': ['Band']},
 'geometry_overlap_resolution': {'stator_minus_windings': True,
  'temporary_winding_tools': ['BooleanTool_Winding_01',
   'BooleanTool_Winding_02',
   'BooleanTool_Winding_03',
   'BooleanTool_Winding_04',
   'BooleanTool_Winding_05',
   'BooleanTool_Winding_06',
   'BooleanTool_Winding_07',
   'BooleanTool_Winding_08',
   'BooleanTool_Winding_09',
   'BooleanTool_Winding_10',
   'BooleanTool_Winding_11',
   'BooleanTool_Winding_12',
   'BooleanTool_Winding_13',
   'BooleanTool_Winding_14',
   'BooleanTool_Winding_15',
   'BooleanTool_Winding_16',
   'BooleanTool_Winding_17',
   'BooleanTool_Winding_18',
   'BooleanTool_Winding_19',
   'BooleanTool_Winding_20',
   'BooleanTool_Winding_21',
   'BooleanTool_Winding_22',
   'BooleanTool_Winding_23',
   'BooleanTool_Winding_24'],
  'deleted_winding_tool_copies': []

In [6]:
# Optional visible solve.
# If RUN_ANALYSIS_IN_NOTEBOOK=True, watch the AEDT progress window while this cell runs.
if RUN_ANALYSIS_IN_NOTEBOOK:
    m2d = getattr(design1, "solver_instance", design1)
    project1.save()
    notebook_analysis_result = m2d.analyze(
        setup=ppt_spec.setup_name,
        cores=NUM_CORES,
        use_auto_settings=False,
    )
    if notebook_analysis_result is False:
        print("PyAEDT returned False. AEDT may still have solved data, so run the export cell to check reports.")
else:
    notebook_analysis_result = "Skipped. Set RUN_ANALYSIS_IN_NOTEBOOK = True in the options cell to solve."

notebook_analysis_result


PyAEDT INFO: Project simulation71 Saved correctly
PyAEDT INFO: Key Desktop/ActiveDSOConfigurations/Maxwell 2D correctly changed.
PyAEDT INFO: Solving design setup PPT_Transient
PyAEDT INFO: Design setup PPT_Transient solved correctly in 0.0h 8.0m 44.0s
PyAEDT INFO: Key Desktop/ActiveDSOConfigurations/Maxwell 2D correctly changed.


True

In [7]:
# Export CSVs and summarize outputs after solve.
# Reports are created before solve when RUN_ANALYSIS_IN_NOTEBOOK=True.
if RUN_ANALYSIS_IN_NOTEBOOK:
    notebook_reports = ppt_setup_result.get("reports", {})
    notebook_exported_reports = export_ppt_reports(design1, project_path, CASE["case_id"])
    notebook_output_summary = summarize_transient_outputs(notebook_exported_reports, ppt_spec)
else:
    notebook_reports = "Skipped. Solve was not requested."
    notebook_exported_reports = {}
    notebook_output_summary = summarize_transient_outputs({}, ppt_spec)

{
    "reports": notebook_reports,
    "exported_reports": notebook_exported_reports,
    "output_summary": notebook_output_summary,
}


{'reports': {'PPT_Phase_Currents': <ansys.aedt.core.visualization.report.standard.Standard at 0x1e6458ab490>,
  'PPT_Torque': <ansys.aedt.core.visualization.report.standard.Standard at 0x1e6458e9850>,
  'PPT_PhaseA_Voltage_Limit': <ansys.aedt.core.visualization.report.standard.Standard at 0x1e64591ee10>,
  'PPT_Phase_Voltages': <ansys.aedt.core.visualization.report.standard.Standard at 0x1e64594b090>,
  'PPT_Losses': <ansys.aedt.core.visualization.report.standard.Standard at 0x1e645986f90>},
 'exported_reports': {'artifact_report_PPT_Phase_Currents': 'Y:\\git\\pyaedt_motor\\simulation\\simulation71\\exports\\notebook_test_10cycle_PPT_Phase_Currents.csv',
  'artifact_report_PPT_Torque': 'Y:\\git\\pyaedt_motor\\simulation\\simulation71\\exports\\notebook_test_10cycle_PPT_Torque.csv',
  'artifact_report_PPT_PhaseA_Voltage_Limit': 'Y:\\git\\pyaedt_motor\\simulation\\simulation71\\exports\\notebook_test_10cycle_PPT_PhaseA_Voltage_Limit.csv',
  'artifact_report_PPT_Phase_Voltages': 'Y:\\git\

In [8]:
# Diagnostic: show recent AEDT Message Manager entries if an error flashed by.
try:
    notebook_aedt_messages = list(desktop.odesktop.GetMessages(project1.name, "IPMSM", 0))
except Exception as exc:
    notebook_aedt_messages = [f"Unable to read AEDT messages: {exc!r}"]

notebook_aedt_messages[-30:]


['Project: simulation71, Design: IPMSM, [info] Normal completion of simulation on server: Local Machine. (12:27:32 AM  Jun 04, 2026)\r\n']

In [9]:
# Optional cleanup cell. Run manually when you are done inspecting AEDT.
# desktop.release_desktop(close_projects=True, close_on_exit=True)
# "AEDT desktop released"
